# Full-Text Search with FTS5

## Quick notes
### What this covers
- Creating FTS5 virtual tables, loading them, and searching with `MATCH`.
- The query language: phrases, prefixes, `NEAR`, `AND` / `OR` / `NOT`, column filters, first-token anchors.
- Ranking with `bm25`, highlighting with `highlight()` and `snippet()`, tokenizers, and external-content tables kept in sync with triggers.
- Maintenance commands, joins, pagination, safe handling of user input, and FTS5 versus `LIKE`.
### Key points
- An FTS5 table is built for "which documents contain these words?". It keeps an inverted index (word to rows) instead of scanning text.
- `WHERE docs MATCH 'query'` searches all columns. `WHERE docs MATCH 'col : word'` limits it to one column.
- `ORDER BY rank` sorts best match first. `rank` is `bm25()` by default, and lower numbers are better (they are negative).
- The tokenizer decides what a "word" is and whether `Café` matches `cafe`, or `running` matches `run`.
- For a big base table use an external-content FTS table so the text is not stored twice, and keep it in sync with triggers.
### Gotchas (the part worth revising twice)
- Raw user text is not safe as a `MATCH` query. Words like `AND`, a lone `"`, a `-` or a `:` cause syntax errors or change the meaning. Quote each term.
- `bm25` and `rank` give smaller (more negative) numbers for better matches, so `ORDER BY rank` ascending is correct.
- FTS5 columns have no types and no constraints. Every column is searchable text.
- Matching is on whole tokens: `cat` does not match `category` unless you write `cat*`.
- `rank`, `highlight()` and `snippet()` only work in a query that has a `MATCH` on that table.
- External-content tables do not update themselves. Forget the triggers and the index goes stale silently.
- The default tokenizer (`unicode61`) does not stem, so `run` will not find `running` unless you choose `porter`.
- `trigram` finds substrings but makes a bigger index and ignores queries shorter than 3 characters.
- FTS5 is a compile-time option. Most builds include it, but check with `PRAGMA compile_options` (look for `ENABLE_FTS5`) on an unfamiliar build.
### When it matters
- Search boxes over articles, tickets, notes, product catalogues, chat logs.
- Anywhere `LIKE '%word%'` gets slow because it must scan every row.
### One-line recap example
- `SELECT title FROM docs WHERE docs MATCH 'sql AND index*' ORDER BY rank LIMIT 10;` means "the 10 best-matching titles containing sql and a word starting with index".

## 01. A first FTS5 table
- `CREATE VIRTUAL TABLE ... USING fts5(col1, col2, ...)` makes a searchable table. Insert into it like a normal table.
- Each column is full-text indexed. Add `UNINDEXED` to a column you want stored but not searched (like an id).
- Seed: small help-centre articles. `article_id` is kept unindexed so we can join back to it.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE VIRTUAL TABLE articles USING fts5(article_id UNINDEXED, title, body);
INSERT INTO articles (article_id, title, body) VALUES
 (1, 'Getting started with SQL',      'SQL is a language for querying relational databases. Start with SELECT and WHERE.'),
 (2, 'Indexes make queries fast',     'An index lets the database find rows without scanning the whole table. Indexes cost write speed.'),
 (3, 'Running backups safely',        'Run a backup before every migration. A backup you never restored is not a backup.'),
 (4, 'Cafe opening hours',            'Our cafe opens at 8. The Café Müller branch opens at 9 and serves breakfast.'),
 (5, 'Database indexing for beginners','Indexing a column helps when you filter or sort by it. SQL databases build indexes as B-trees.'),
 (6, 'Troubleshooting slow queries',  'Slow queries often lack an index. Check the query plan, then add an index or rewrite the query.'),
 (7, 'Category management',           'Categories group products. A category can have many products and a product has one category.');
SELECT article_id, title FROM articles ORDER BY article_id;

article_id | title                          
-----------+--------------------------------
1          | Getting started with SQL       
2          | Indexes make queries fast      
3          | Running backups safely         
4          | Cafe opening hours             
5          | Database indexing for beginners
6          | Troubleshooting slow queries   
7          | Category management            
(7 rows)



## 02. MATCH basics
- `MATCH 'word'` searches every indexed column. The search ignores case.
- Two words with a space between them mean `AND`: both must appear (in any columns, in any order).
- Without `ORDER BY`, the order is not guaranteed, so always add one.

In [3]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH 'index' ORDER BY article_id;

article_id | title                       
-----------+-----------------------------
2          | Indexes make queries fast   
6          | Troubleshooting slow queries
(2 rows)



In [4]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH 'sql databases' ORDER BY article_id;

article_id | title                          
-----------+--------------------------------
1          | Getting started with SQL       
5          | Database indexing for beginners
(2 rows)



## 03. Whole tokens, prefixes and the star
- `index` matches the token `index` only, not `indexes` or `indexing`. Add `*` for a prefix: `index*`.
- `cat` finds nothing here, but `cat*` finds "category" and "categories".
- Prefix queries are fast enough on small tables; on large ones add a `prefix='2 3'` option to the table.

In [5]:
%%sql
SELECT 'index'  AS query, count(*) AS hits FROM articles WHERE articles MATCH 'index'
UNION ALL SELECT 'index*', count(*) FROM articles WHERE articles MATCH 'index*'
UNION ALL SELECT 'cat',    count(*) FROM articles WHERE articles MATCH 'cat'
UNION ALL SELECT 'cat*',   count(*) FROM articles WHERE articles MATCH 'cat*'
ORDER BY query;

query  | hits
-------+-----
cat    | 0   
cat*   | 1   
index  | 2   
index* | 3   
(4 rows)



In [6]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH 'index*' ORDER BY article_id;

article_id | title                          
-----------+--------------------------------
2          | Indexes make queries fast      
5          | Database indexing for beginners
6          | Troubleshooting slow queries   
(3 rows)



## 04. Phrases
- Double quotes make a phrase: the words must be next to each other, in that order.
- Inside a SQL string, the phrase quotes are plain `"` characters: `'"query plan"'`.
- Compare the phrase with the same words unquoted.

In [7]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH '"query plan"' ORDER BY article_id;

article_id | title                       
-----------+-----------------------------
6          | Troubleshooting slow queries
(1 row)



In [8]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH '"plan query"' ORDER BY article_id;

article_id | title
-----------+------
(0 rows)



## 05. Boolean operators: AND, OR, NOT
- Operators must be UPPERCASE. A lowercase `or` is treated as an ordinary word.
- `NOT` here is a binary operator: `a NOT b` means "a but not b". It cannot start a query.
- Use parentheses to group. `AND` binds tighter than `OR`, so be explicit.

In [9]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH 'index* NOT sql' ORDER BY article_id;

article_id | title                       
-----------+-----------------------------
2          | Indexes make queries fast   
6          | Troubleshooting slow queries
(2 rows)



In [10]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH '(backup OR breakfast) NOT migration' ORDER BY article_id;

article_id | title             
-----------+-------------------
4          | Cafe opening hours
(1 row)



In [11]:
%%sql expect_error
SELECT article_id FROM articles WHERE articles MATCH 'NOT sql' ORDER BY article_id;

ERROR: fts5: syntax error near "NOT"



In [12]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH 'backup OR breakfast' ORDER BY article_id;

article_id | title                 
-----------+-----------------------
3          | Running backups safely
4          | Cafe opening hours    
(2 rows)



## 06. NEAR
- `NEAR(a b, N)` means the words occur within N tokens of each other (default N is 10).
- Useful for "words about the same thing, not necessarily adjacent".
- In article 6's body, `slow` is the first token and `plan` is the tenth, so they are 8 tokens apart. A distance of 10 finds it; a distance of 2 does not.
- `NEAR` only counts tokens inside one column.

In [13]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH 'NEAR(slow plan, 10)' ORDER BY article_id;

article_id | title                       
-----------+-----------------------------
6          | Troubleshooting slow queries
(1 row)



In [14]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH 'NEAR(slow plan, 2)' ORDER BY article_id;

article_id | title
-----------+------
(0 rows)



## 07. Column filters and the first-token anchor
- `title : word` limits a term to one column. `{title body} : word` limits it to several. `- title : word` means "every column except title".
- A leading `^` means "must be the first token of the column": `^getting` finds titles that start with it.
- Column filters are how you make title hits matter more than body hits.

In [15]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH 'title : index*' ORDER BY article_id;

article_id | title                          
-----------+--------------------------------
2          | Indexes make queries fast      
5          | Database indexing for beginners
(2 rows)



In [16]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH '- title : index*' ORDER BY article_id;

article_id | title                          
-----------+--------------------------------
2          | Indexes make queries fast      
5          | Database indexing for beginners
6          | Troubleshooting slow queries   
(3 rows)



In [17]:
%%sql
SELECT article_id, title FROM articles WHERE articles MATCH 'title : ^database' ORDER BY article_id;

article_id | title                          
-----------+--------------------------------
5          | Database indexing for beginners
(1 row)



## 08. Ranking with bm25 and ORDER BY rank
- `rank` is a hidden column. By default it equals `bm25(articles)`. More relevant rows get a lower (more negative) number.
- bm25 rewards rare words, repeated words and short documents.
- Always sort with `ORDER BY rank`, then add a tie-breaker such as `article_id` so the order is stable.

In [18]:
%%sql
SELECT article_id, title, round(rank, 4) AS score
FROM articles
WHERE articles MATCH 'index OR indexes OR indexing'
ORDER BY rank, article_id;

article_id | title                           | score  
-----------+---------------------------------+--------
5          | Database indexing for beginners | -2.6711
2          | Indexes make queries fast       | -1.8402
6          | Troubleshooting slow queries    | -1.053 
(3 rows)



## 09. Column weights in bm25
- `bm25(table, w1, w2, ...)` gives a weight to each column, in table order (including `UNINDEXED` ones, which should get 0).
- Here: article_id 0, title 10, body 1. A hit in the title now counts ten times more than the same hit in the body.
- You can use this in `ORDER BY`, or set it permanently: `INSERT INTO articles(articles, rank) VALUES('rank', 'bm25(0.0, 10.0, 1.0)')`.

In [19]:
%%sql
SELECT article_id, title, round(bm25(articles, 0.0, 10.0, 1.0), 4) AS weighted_score
FROM articles
WHERE articles MATCH 'index* OR queries'
ORDER BY bm25(articles, 0.0, 10.0, 1.0), article_id;

article_id | title                           | weighted_score
-----------+---------------------------------+---------------
2          | Indexes make queries fast       | -2.0431       
6          | Troubleshooting slow queries    | -1.8876       
5          | Database indexing for beginners | -0.4973       
(3 rows)



## 10. highlight() and snippet()
- `highlight(table, col_index, open, close)` returns the full column text with matches wrapped.
- `snippet(table, col_index, open, close, ellipsis, max_tokens)` returns a short excerpt around the match. Column index counts from 0 (`article_id` is 0, `title` 1, `body` 2). Use `-1` to let FTS5 pick the best column.
- Both only work when the query has a `MATCH`.

In [20]:
%%sql
SELECT article_id,
       highlight(articles, 1, '[', ']') AS title_hl,
       snippet(articles, 2, '<<', '>>', '...', 8) AS body_snippet
FROM articles
WHERE articles MATCH 'index*'
ORDER BY rank, article_id;

article_id | title_hl                          | body_snippet                                       
-----------+-----------------------------------+----------------------------------------------------
2          | [Indexes] make queries fast       | An <<index>> lets the database find rows without...
5          | Database [indexing] for beginners | ...SQL databases build <<indexes>> as B-trees.     
6          | Troubleshooting slow queries      | Slow queries often lack an <<index>>. Check the... 
(3 rows)



## 11. Tokenizers: unicode61, porter, trigram
- `unicode61` (default) splits on non-letters and lowercases. It does not stem. By default it removes diacritics (`remove_diacritics=1`), so `Café` matches `cafe`.
- `porter` is a stemmer on top of another tokenizer: `tokenize = 'porter unicode61'`. `running` and `run` become the same token.
- `trigram` indexes every 3-letter piece, so substring searches like `ndex` work. Add `case_sensitive 0` (the default) and `remove_diacritics 1` as needed.
- Build the same text with each tokenizer and compare.

In [21]:
%%sql
CREATE VIRTUAL TABLE t_plain  USING fts5(txt, tokenize = 'unicode61');
CREATE VIRTUAL TABLE t_keep   USING fts5(txt, tokenize = 'unicode61 remove_diacritics 0');
CREATE VIRTUAL TABLE t_porter USING fts5(txt, tokenize = 'porter unicode61');
CREATE VIRTUAL TABLE t_tri    USING fts5(txt, tokenize = 'trigram');
INSERT INTO t_plain  VALUES ('Running late at the Café'), ('The runner indexes pages');
INSERT INTO t_keep   VALUES ('Running late at the Café'), ('The runner indexes pages');
INSERT INTO t_porter VALUES ('Running late at the Café'), ('The runner indexes pages');
INSERT INTO t_tri    VALUES ('Running late at the Café'), ('The runner indexes pages');
SELECT 'plain: run'          AS test, count(*) AS hits FROM t_plain  WHERE t_plain  MATCH 'run'
UNION ALL SELECT 'plain: running',    count(*) FROM t_plain  WHERE t_plain  MATCH 'running'
UNION ALL SELECT 'porter: run',       count(*) FROM t_porter WHERE t_porter MATCH 'run'
UNION ALL SELECT 'porter: index',     count(*) FROM t_porter WHERE t_porter MATCH 'index'
UNION ALL SELECT 'plain: cafe',       count(*) FROM t_plain  WHERE t_plain  MATCH 'cafe'
UNION ALL SELECT 'keep accents: cafe',count(*) FROM t_keep   WHERE t_keep   MATCH 'cafe'
UNION ALL SELECT 'trigram: ndex',     count(*) FROM t_tri    WHERE t_tri    MATCH 'ndex'
UNION ALL SELECT 'plain: ndex',       count(*) FROM t_plain  WHERE t_plain  MATCH 'ndex'
ORDER BY test;

test               | hits
-------------------+-----
keep accents: cafe | 0   
plain: cafe        | 1   
plain: ndex        | 0   
plain: run         | 0   
plain: running     | 1   
porter: index      | 1   
porter: run        | 1   
trigram: ndex      | 1   
(8 rows)



In [22]:
%%sql
SELECT 'trigram: 2 chars "ru" (matches nothing, too short)' AS test, count(*) AS hits FROM t_tri WHERE t_tri MATCH '"ru"';

test                                               | hits
---------------------------------------------------+-----
trigram: 2 chars "ru" (matches nothing, too short) | 0   
(1 row)



## 12. A search box that returns the base row: joining FTS to a table
- Keep real data in a normal table and the FTS table beside it; join on the id.
- The FTS table can be the driver (filter by `MATCH`) and the base table supplies the other columns.
- Below: `products` has a price and a category; the FTS index covers only name and description.

In [23]:
%%sql
CREATE TABLE products (
    product_id  INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    description TEXT NOT NULL,
    category    TEXT NOT NULL,
    price       REAL NOT NULL
);
INSERT INTO products VALUES
 (1,'Trail Running Shoes','Light shoes for running on mud and rock.','footwear',89.0),
 (2,'City Walking Shoes','Comfortable everyday shoes with soft soles.','footwear',65.5),
 (3,'Rain Jacket','Waterproof jacket, packs into its own pocket.','clothing',120.0),
 (4,'Running Socks','Cushioned socks that wick sweat during long runs.','clothing',12.0),
 (5,'Water Bottle','Steel bottle keeps drinks cold for a day.','gear',18.0);
CREATE VIRTUAL TABLE products_fts USING fts5(name, description, content='products', content_rowid='product_id', tokenize='porter unicode61');
INSERT INTO products_fts(products_fts) VALUES ('rebuild');
SELECT p.product_id, p.name, p.category, p.price
FROM products_fts f JOIN products p ON p.product_id = f.rowid
WHERE products_fts MATCH 'run'
ORDER BY f.rank, p.product_id;

product_id | name                | category | price
-----------+---------------------+----------+------
4          | Running Socks       | clothing | 12.0 
1          | Trail Running Shoes | footwear | 89.0 
(2 rows)



## 13. External-content tables and triggers
- `content='products'` means the FTS table stores only the index, not a copy of the text. It reads text from `products` when it needs it (for `highlight()`, `snippet()`, and rebuilds).
- You must keep the index in sync yourself. The standard recipe is three triggers (insert, delete, update) using the special `'delete'` command.
- On delete or update, the old values must be given to the FTS table so it can remove the old tokens.
- See 05-DDL-And-Integrity/02-Views-And-Triggers for trigger basics.

In [24]:
%%sql
CREATE TRIGGER products_ai AFTER INSERT ON products BEGIN
    INSERT INTO products_fts(rowid, name, description) VALUES (new.product_id, new.name, new.description);
END;
CREATE TRIGGER products_ad AFTER DELETE ON products BEGIN
    INSERT INTO products_fts(products_fts, rowid, name, description) VALUES ('delete', old.product_id, old.name, old.description);
END;
CREATE TRIGGER products_au AFTER UPDATE ON products BEGIN
    INSERT INTO products_fts(products_fts, rowid, name, description) VALUES ('delete', old.product_id, old.name, old.description);
    INSERT INTO products_fts(rowid, name, description) VALUES (new.product_id, new.name, new.description);
END;

In [25]:
%%sql
INSERT INTO products VALUES (6,'Running Hat','Sun hat for runners with a mesh back.','clothing',22.0);
UPDATE products SET description = 'Steel bottle, dishwasher safe, keeps drinks cold.' WHERE product_id = 5;
DELETE FROM products WHERE product_id = 2;
SELECT p.product_id, p.name FROM products_fts f JOIN products p ON p.product_id = f.rowid
WHERE products_fts MATCH 'run OR dishwasher OR walking'
ORDER BY p.product_id;

product_id | name               
-----------+--------------------
1          | Trail Running Shoes
4          | Running Socks      
5          | Water Bottle       
6          | Running Hat        
(4 rows)



## 14. What happens without the triggers
- If the base table changes and the index does not, searches show stale results.
- Here we bypass the triggers by dropping one, change a row, and search for the new text.
- The `'rebuild'` command re-reads everything from the content table and fixes it.

In [26]:
%%sql
DROP TRIGGER products_au;
UPDATE products SET description = 'Steel bottle with a carabiner clip.' WHERE product_id = 5;
SELECT 'carabiner (new text)' AS query, count(*) AS hits FROM products_fts WHERE products_fts MATCH 'carabiner'
UNION ALL SELECT 'dishwasher (old text)', count(*) FROM products_fts WHERE products_fts MATCH 'dishwasher'
ORDER BY query;

query                 | hits
----------------------+-----
carabiner (new text)  | 0   
dishwasher (old text) | 1   
(2 rows)



In [27]:
%%sql
INSERT INTO products_fts(products_fts) VALUES ('rebuild');
SELECT 'carabiner (new text)' AS query, count(*) AS hits FROM products_fts WHERE products_fts MATCH 'carabiner'
UNION ALL SELECT 'dishwasher (old text)', count(*) FROM products_fts WHERE products_fts MATCH 'dishwasher'
ORDER BY query;

query                 | hits
----------------------+-----
carabiner (new text)  | 1   
dishwasher (old text) | 0   
(2 rows)



## 15. Contentless tables
- `content=''` stores no text at all, only the index. Smallest size, but you can read back only the rowid. Selecting the text column returns NULL (last cell below).
- `highlight()` and `snippet()` cannot work (there is no text), and normal `DELETE` / `UPDATE` are not allowed unless you add `contentless_delete=1` (3.43+).
- Good when the original text lives somewhere else (files, another system) and you only need "which ids match".

In [28]:
%%sql
CREATE VIRTUAL TABLE notes_idx USING fts5(body, content='', contentless_delete=1);
INSERT INTO notes_idx(rowid, body) VALUES (101, 'buy oat milk and bread'), (102, 'call the plumber about the tap'), (103, 'bread recipe from grandma');
SELECT rowid FROM notes_idx WHERE notes_idx MATCH 'bread' ORDER BY rowid;

rowid
-----
101  
103  
(2 rows)



In [29]:
%%sql
DELETE FROM notes_idx WHERE rowid = 101;
SELECT rowid FROM notes_idx WHERE notes_idx MATCH 'bread' ORDER BY rowid;

rowid
-----
103  
(1 row)



In [30]:
%%sql
SELECT rowid, body FROM notes_idx WHERE notes_idx MATCH 'bread' ORDER BY rowid;

rowid | body
------+-----
103   | NULL
(1 row)



## 16. Maintenance: optimize, integrity-check, rebuild
- `INSERT INTO t(t) VALUES('optimize')` merges index segments into one. Do it after bulk loads.
- `INSERT INTO t(t) VALUES('integrity-check')` checks the index for internal damage. For an external-content table, pass `rank` = 1 (`INSERT INTO t(t, rank) VALUES('integrity-check', 1)`) so it also compares the index with the real text in the content table. That form catches missing triggers; in the demo it fails with the generic message "database disk image is malformed".
- `INSERT INTO t(t) VALUES('rebuild')` recreates the index from the content.
- `INSERT INTO t(t, rank) VALUES('automerge', N)` and `'usermerge'` tune merging; defaults are usually fine.

In [31]:
%%sql
INSERT INTO products_fts(products_fts) VALUES ('optimize');
INSERT INTO products_fts(products_fts, rank) VALUES ('integrity-check', 1);
SELECT 'index consistent' AS result;

result          
----------------
index consistent
(1 row)



In [32]:
%%sql
-- Break the index on purpose: change the base table with the update trigger gone, then check.
UPDATE products SET name = 'Steel Flask' WHERE product_id = 5;

In [33]:
%%sql expect_error
INSERT INTO products_fts(products_fts, rank) VALUES ('integrity-check', 1);

ERROR: database disk image is malformed



In [34]:
%%sql
INSERT INTO products_fts(products_fts) VALUES ('rebuild');
INSERT INTO products_fts(products_fts, rank) VALUES ('integrity-check', 1);
SELECT p.product_id, p.name FROM products_fts f JOIN products p ON p.product_id = f.rowid WHERE products_fts MATCH 'flask' ORDER BY p.product_id;

product_id | name       
-----------+------------
5          | Steel Flask
(1 row)



## 17. Pagination
- Use `LIMIT` and `OFFSET` with a fully deterministic `ORDER BY rank, id`.
- `OFFSET` still computes the skipped rows, so very deep pages get slower. For a "next page" button, remember the last `(rank, id)` pair and filter after it (keyset paging).
- Total result count for a pager: `SELECT count(*) FROM t WHERE t MATCH ...`.

In [35]:
%%sql
SELECT article_id, title, round(rank, 3) AS score
FROM articles
WHERE articles MATCH 'index* OR quer*'
ORDER BY rank, article_id
LIMIT 2 OFFSET 0;

article_id | title                        | score 
-----------+------------------------------+-------
6          | Troubleshooting slow queries | -0.753
2          | Indexes make queries fast    | -0.637
(2 rows)



In [36]:
%%sql
SELECT article_id, title, round(rank, 3) AS score
FROM articles
WHERE articles MATCH 'index* OR quer*'
ORDER BY rank, article_id
LIMIT 2 OFFSET 2;

article_id | title                           | score 
-----------+---------------------------------+-------
5          | Database indexing for beginners | -0.382
1          | Getting started with SQL        | -0.263
(2 rows)



## 18. Escaping user input
- A search box passes whatever the user typed. `MATCH` has its own syntax, so this can fail or mean something else.
- Safe rule: split the input into words, wrap each in double quotes (doubling any `"` inside), then join with spaces (implicit AND). A quoted `"index*"` is a literal word, so the star is dropped and prefix search is lost; to allow prefix search, append `*` after the closing quote yourself (`"index"*`).
- Always use a bound parameter for the final string, never string-pasting it into SQL (see 08-SQLite-Specials/04-SQLite-With-Python, section 03, for injection notes). The escaping below happens in Python, with a parameter.

In [37]:
%%sql expect_error
SELECT count(*) FROM articles WHERE articles MATCH 'what is "an index';

ERROR: unterminated string



In [38]:
%%sql expect_error
SELECT count(*) FROM articles WHERE articles MATCH 'sql-index';

ERROR: no such column: index



In [39]:
def fts_query(user_text):
    words = [w for w in user_text.split() if w]
    return " ".join('"' + w.replace('"', '""') + '"' for w in words)

for raw in ['what is "an index', 'sql-index', 'AND', 'index*']:
    q = fts_query(raw)
    n = con.execute("SELECT count(*) FROM articles WHERE articles MATCH ?", (q,)).fetchone()[0]
    print(f"{raw!r:24} -> {q!r:30} hits={n}")

'what is "an index'      -> '"what" "is" """an" "index"'   hits=0
'sql-index'              -> '"sql-index"'                  hits=0
'AND'                    -> '"AND"'                        hits=3
'index*'                 -> '"index*"'                     hits=2


## 19. FTS5 versus LIKE: the plans
- `LIKE '%word%'` cannot use a normal index (the leading wildcard), so it scans every row and every character.
- An FTS5 `MATCH` looks the word up in the inverted index and reads only the matching rows.
- The plan wording is the clue: `SCAN` versus `SCAN ... VIRTUAL TABLE INDEX`. Also `LIKE` is a substring test, so `'%index%'` happens to hit `indexes` and `indexing` too, with no ranking.

In [40]:
%%sql
CREATE TABLE articles_plain AS SELECT article_id, title, body FROM articles;
EXPLAIN QUERY PLAN SELECT article_id FROM articles_plain WHERE body LIKE '%index%';

id | parent | notused | detail             
---+--------+---------+--------------------
2  | 0      | 0       | SCAN articles_plain
(1 row)



In [41]:
%%sql
EXPLAIN QUERY PLAN SELECT article_id FROM articles WHERE articles MATCH 'index';

id | parent | notused | detail                                
---+--------+---------+---------------------------------------
2  | 0      | 0       | SCAN articles VIRTUAL TABLE INDEX 0:M3
(1 row)



In [42]:
%%sql
SELECT 'LIKE %index%' AS method, count(*) AS hits FROM articles_plain WHERE body LIKE '%index%' OR title LIKE '%index%'
UNION ALL SELECT 'MATCH index*', count(*) FROM articles WHERE articles MATCH 'index*'
UNION ALL SELECT 'MATCH index',  count(*) FROM articles WHERE articles MATCH 'index'
ORDER BY method;

method       | hits
-------------+-----
LIKE %index% | 3   
MATCH index  | 2   
MATCH index* | 3   
(3 rows)



## 20. Looking inside: fts5vocab
- `fts5vocab` is a virtual table that exposes the index vocabulary: which tokens exist and how often.
- Types: `row` (term, doc, cnt), `col` (per column), `instance` (every occurrence).
- Use it for autocomplete, typo suggestions, or to see what the tokenizer really produced.

In [43]:
%%sql
CREATE VIRTUAL TABLE articles_vocab USING fts5vocab(articles, row);
SELECT term, doc, cnt FROM articles_vocab WHERE term LIKE 'index%' ORDER BY term;

term     | doc | cnt
---------+-----+----
index    | 2   | 3  
indexes  | 2   | 3  
indexing | 1   | 2  
(3 rows)



In [44]:
%%sql
-- the most widespread tokens: in how many articles does each appear?
SELECT term, doc FROM articles_vocab ORDER BY doc DESC, term LIMIT 5;

term     | doc
---------+----
a        | 4  
and      | 3  
the      | 3  
an       | 2  
database | 2  
(5 rows)



## 21. Limits and more gotchas
- Every column is TEXT-like and unconstrained. A `rowid` is the only key. Query it as `rowid`.
- You cannot add an ordinary index, a foreign key or a `CHECK` on an FTS5 table, and `ALTER TABLE ADD COLUMN` is not supported. Rebuild the table to change columns.
- Only `MATCH`, `=` on rowid and full scans are fast. Ordering by anything other than `rank` or `rowid` makes SQLite sort the matched rows afterwards.
- Unquoted words must be letters and digits (and `_`); anything else needs double quotes. `name:foo` and `-foo` have special meaning.
- The `UNINDEXED` column is stored but not searchable, so `MATCH 'article_id : 3'` finds nothing.
- Prefix search on a large table benefits from `prefix='2 3'` in the table definition.

In [45]:
%%sql expect_error
ALTER TABLE articles ADD COLUMN extra TEXT;

ERROR: virtual tables may not be altered



In [46]:
%%sql
SELECT count(*) AS hits FROM articles WHERE articles MATCH 'article_id : 3';

hits
----
0   
(1 row)

